# 🔧 Aplicaciones Prácticas con Modelos de Difusión

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mattbarreto/ifts24-lab-pdi-2026/blob/master/009%20-%20modelos_difusion/03_Aplicaciones_Practicas_Difusion.ipynb)

**Procesamiento Digital de Imágenes — IFTS24**
Prof. Matias Barreto — 2026

**Unidad 9 · Bloque 3 — 60 minutos**

---

## Al terminar este bloque vas a poder:

1. Aplicar la técnica de inpainting para rellenar regiones de una imagen de forma coherente.
2. Usar image-to-image para transformar imágenes existentes a partir de un prompt de texto.
3. Controlar el parámetro `strength` y entender su efecto sobre el resultado final.
4. Guardar imágenes generadas con metadata que registre el prompt y la técnica usada.

---

### ✎ Para pensar

1. ¿Cómo sabe el modelo qué poner en la región borrada? ¿Usa solo el prompt o también el contexto visual de la imagen?
2. Si la máscara cubre el 90% de la imagen, ¿el resultado se parece más a un inpainting o a una generación desde cero? ¿Por qué?

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Inpainting** | Técnica para rellenar regiones borradas o seleccionadas de una imagen con contenido coherente con el contexto visual. |
| **Outpainting** | Extensión de una imagen más allá de sus bordes originales, manteniendo coherencia visual y estilística. |
| **Image-to-image** | Modalidad de generación que parte de una imagen existente y la transforma según un prompt de texto. |
| **Mask** | Imagen binaria que indica qué regiones modificar (blanco) y cuáles conservar sin cambios (negro). |
| **Strength** | Parámetro entre 0 y 1 que controla cuánto se aleja el resultado de la imagen de entrada; 0 = sin cambios, 1 = imagen completamente nueva. |
| **Pipeline** | Objeto de `diffusers` que encapsula todo el proceso de generación preconfigurado para un caso de uso específico. |

## 1. Configuración inicial

Primero instalamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Instalación de librerías
# Esto puede tardar 2-3 minutos la primera vez
!pip install -q diffusers transformers accelerate torch pillow matplotlib requests

In [ ]:
# Imports necesarios
import torch
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO
import numpy as np

# Verificamos si tenemos GPU disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

if device == "cpu":
    print("\n⚠️ Advertencia: Estás usando CPU. Los procesos van a ser más lentos.")
    print("   En Google Colab, podés activar GPU en: Runtime → Change runtime type → GPU")

# Configuración para gráficos
plt.rcParams['figure.figsize'] = (15, 5)

## 2. Funciones auxiliares

Estas funciones nos van a ayudar a cargar imágenes y visualizar resultados fácilmente.

In [ ]:
def cargar_imagen_desde_url(url, tamaño=None):
    """
    Descarga una imagen desde una URL y opcionalmente la redimensiona.

    Parámetros:
    - url: dirección web de la imagen
    - tamaño: tupla (ancho, alto) para redimensionar. None mantiene tamaño original
    """
    response = requests.get(url)
    imagen = Image.open(BytesIO(response.content)).convert('RGB')

    if tamaño:
        imagen = imagen.resize(tamaño)

    return imagen


def mostrar_imagenes(imagenes, titulos):
    """
    Muestra múltiples imágenes lado a lado con sus títulos.

    Parámetros:
    - imagenes: lista de imágenes PIL
    - titulos: lista de strings con los títulos
    """
    n = len(imagenes)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))

    if n == 1:
        axes = [axes]

    for img, titulo, ax in zip(imagenes, titulos, axes):
        ax.imshow(img)
        ax.set_title(titulo, fontsize=12)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# Ejemplo: crear una máscara rectangular simple
def crear_mascara_rectangular(ancho, alto, x, y, ancho_rect, alto_rect):
    """
    Crea una máscara con un rectángulo blanco sobre fondo negro.

    Parámetros:
    - ancho, alto: dimensiones de la imagen
    - x, y: esquina superior izquierda del rectángulo
    - ancho_rect, alto_rect: dimensiones del rectángulo
    """
    mascara_array = np.zeros((alto, ancho), dtype=np.uint8)
    mascara_array[y:y+alto_rect, x:x+ancho_rect] = 255
    return Image.fromarray(mascara_array)


print("Funciones auxiliares cargadas correctamente.")

---

## 3. Aplicación 1: Inpainting (Eliminar objetos de fotos)

### ¿Qué es inpainting?

Es el proceso de **rellenar o eliminar partes de una imagen**. Muy útil para:
- Remover objetos no deseados (personas, cables, manchas)
- Restaurar partes dañadas de fotos antiguas
- Completar regiones faltantes

### ¿Cómo funciona?

1. Tenemos una **imagen original**
2. Creamos una **máscara** (imagen en blanco y negro donde el blanco indica qué queremos eliminar)
3. El modelo **rellena la región enmascarada** con contenido coherente

**Conexión con lo que ya vimos**: Recordá cuando trabajamos con segmentación y máscaras. Aquí usamos el mismo concepto, pero en vez de separar regiones, las rellenamos.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

# Cargamos el modelo de inpainting
# Este paso descarga el modelo (puede tardar unos minutos la primera vez)
print("Cargando modelo de inpainting...")
pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",  # Modelo pre-entrenado
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,  # Precisión (float16 es más rápido en GPU)
    safety_checker=None  # Desactivamos verificador de seguridad para simplificar
)
pipe_inpaint = pipe_inpaint.to(device)
print("Modelo cargado exitosamente.")

### Ejemplo práctico: Eliminar un objeto de una foto

Vamos a usar una foto de ejemplo y crear una máscara para eliminar un objeto.

In [ ]:
# ============================================
# COMPLETÁ AQUÍ CON TU PROPIA IMAGEN
# ============================================
#
# Tipo de imagen recomendada para INPAINTING:
# - Una foto de exteriores (parque, calle, playa, jardín)
# - Con objetos claramente definidos que podrías querer eliminar
# - Ejemplos: banco en un parque, persona en paisaje, auto en calle
# - Resolución recomendada: al menos 512x512 píxeles
#
# Dónde buscar imágenes:
# - Wikimedia Commons (libres de derechos)
# - Unsplash.com (fotos gratuitas de alta calidad)
# - Pexels.com (fotos gratuitas)
#
# IMPORTANTE: Copiá la URL directa a la imagen (debe terminar en .jpg, .png, etc.)

# Completá tu URL aquí:
url_imagen = "https://www.chapultepec.org.mx/wp-content/uploads/2021/11/zorro4-2.jpg"  # 👈 Pegá tu URL entre las comillas

# Cargamos la imagen
imagen_original = cargar_imagen_desde_url(url_imagen, tamaño=(512, 512))

# Para este ejemplo, vamos a crear una máscara rectangular simple
# La máscara indica qué parte de la imagen queremos eliminar/rellenar
# Ajustá los valores para cubrir el objeto que querés eliminar

mascara = crear_mascara_rectangular(
    ancho=512,
    alto=512,
    x=200,      # 👈 Posición horizontal donde empieza el rectángulo (modificable)
    y=350,      # 👈 Posición vertical donde empieza el rectángulo (modificable)
    ancho_rect=200,  # 👈 Ancho del área a eliminar (modificable)
    alto_rect=200    # 👈 Alto del área a eliminar (modificable)
)

# Visualizamos
mostrar_imagenes(
    [imagen_original, mascara],
    ['Imagen Original', 'Máscara (blanco = eliminar)']
)

In [ ]:
# Aplicamos inpainting
# prompt: descripción de qué queremos en la región enmascarada
# num_inference_steps: cantidad de pasos del proceso (más pasos = mejor calidad pero más lento)
# guidance_scale: qué tan fuerte seguir el prompt (valores típicos: 7-15)

print("Procesando... esto puede tardar 10-30 segundos")

# 👇 AJUSTÁ este prompt según lo que querés que aparezca en el área eliminada
# Debe describir algo coherente con el resto de la imagen
# Ejemplos: "green grass field", "blue sky with clouds", "stone pavement", "trees and bushes"
prompt_relleno = "stone pavement"  # 👈 Modificá esto según tu imagen y lo que querés lograr

resultado_inpaint = pipe_inpaint(
    prompt=prompt_relleno,
    image=imagen_original,
    mask_image=mascara,
    num_inference_steps=100,  # Pasos de difusión (20-50 es un buen rango)
    guidance_scale=7.5  # Cuánto seguir el prompt (5-15 típicamente)
).images[0]

print("Proceso completado.")

# Mostramos el resultado
mostrar_imagenes(
    [imagen_original, mascara, resultado_inpaint],
    ['Original', 'Máscara', 'Resultado']
)

### Experimentación: Probá con diferentes prompts

El prompt controla qué aparece en la región enmascarada. Probá cambiar el texto para ver diferentes resultados.

**Consejos para escribir buenos prompts:**
- Describí algo coherente con el resto de la imagen
- Sé específico pero conciso
- Pensá en qué debería verse natural en ese lugar

**Ejemplos según el tipo de imagen:**
- Si tenés un parque: "green grass field", "stone path", "flower garden"
- Si tenés una playa: "blue ocean water", "sandy beach", "palm trees"
- Si tenés una calle: "empty sidewalk", "brick wall", "building facade"
- Si tenés el cielo: "blue sky with clouds", "clear sky", "sunset colors"

**Ejercicio**: Ejecutá la celda anterior varias veces con diferentes prompts. ¿Cómo cambia el resultado?

### Crear tu propia máscara

Para experimentar con tus propias imágenes, podés crear máscaras simples con código.

In [ ]:
# Ejemplo: crear una máscara rectangular simple
def crear_mascara_rectangular(ancho, alto, x, y, ancho_rect, alto_rect):
    """
    Crea una máscara con un rectángulo blanco sobre fondo negro.

    Parámetros:
    - ancho, alto: dimensiones de la imagen
    - x, y: esquina superior izquierda del rectángulo
    - ancho_rect, alto_rect: dimensiones del rectángulo
    """
    mascara_array = np.zeros((alto, ancho), dtype=np.uint8)
    mascara_array[y:y+alto_rect, x:x+ancho_rect] = 255
    return Image.fromarray(mascara_array)

# Ejemplo de uso
mascara_prueba = crear_mascara_rectangular(
    ancho=512,
    alto=512,
    x=150,  # posición x del rectángulo (modificable)
    y=150,  # posición y del rectángulo (modificable)
    ancho_rect=200,  # ancho del rectángulo (modificable)
    alto_rect=200  # alto del rectángulo (modificable)
)

plt.imshow(mascara_prueba, cmap='gray')
plt.title('Máscara personalizada')
plt.axis('off')
plt.show()

---

## 4. Aplicación 2: Super-Resolution (Mejorar calidad de imágenes)

### ¿Qué es super-resolution?

Es el proceso de **mejorar la resolución y calidad** de una imagen de baja calidad. Útil para:
- Mejorar fotos pixeladas o de baja resolución
- Recuperar detalles perdidos por compresión
- Mejorar capturas de video de baja calidad

### Diferencia con interpolación tradicional

La interpolación (como la que vimos antes) **estima píxeles intermedios** promediando valores vecinos.

Super-resolution con difusión **reconstruye detalles reales** usando conocimiento aprendido de miles de imágenes de alta calidad.

In [ ]:
from diffusers import StableDiffusionUpscalePipeline

# Cargamos el modelo de super-resolution
print("Cargando modelo de super-resolution...")
pipe_upscale = StableDiffusionUpscalePipeline.from_pretrained(
    "stabilityai/stable-diffusion-x4-upscaler",  # Modelo que amplifica 4x
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe_upscale = pipe_upscale.to(device)
print("Modelo cargado exitosamente.")

### Ejemplo práctico: Mejorar una imagen de baja resolución

In [ ]:
# ============================================
# COMPLETÁ AQUÍ CON TU PROPIA IMAGEN
# ============================================
#
# Tipo de imagen recomendada para SUPER-RESOLUTION:
# - Foto con buenos detalles (rostro, animal, objeto con textura)
# - Preferiblemente con un sujeto principal claro
# - Ejemplos: retrato de persona, foto de mascota, flor en primer plano
# - No importa si es de baja resolución, la vamos a reducir igual
#
# Dónde buscar imágenes:
# - Wikimedia Commons (libres de derechos)
# - Unsplash.com (fotos gratuitas)
# - Pexels.com (fotos gratuitas)
#
# IMPORTANTE: Copiá la URL directa a la imagen (debe terminar en .jpg, .png, etc.)

# Completá tu URL aquí:
url_imagen_hq = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTy0FbWVyNbCsr4BEe7ya41Nte03L75MLqtxA&s"  # 👈 Pegá tu URL entre las comillas

# Cargamos la imagen y la preparamos
imagen_alta = cargar_imagen_desde_url(url_imagen_hq, tamaño=(512, 512))

# Simulamos una imagen de baja resolución (reducimos a 128x128)
imagen_baja = imagen_alta.resize((128, 128), Image.Resampling.BILINEAR)

# La volvemos a 512x512 para ver cómo queda con interpolación simple
imagen_interpolada = imagen_baja.resize((512, 512), Image.Resampling.BILINEAR)

mostrar_imagenes(
    [imagen_baja, imagen_interpolada],
    ['Baja Resolución (128x128)', 'Interpolación Simple (512x512)']
)

In [ ]:
# Aplicamos super-resolution con modelo de difusión
# prompt: descripción general de la imagen para guiar el proceso
# num_inference_steps: cantidad de pasos (más pasos = mejor calidad)

print("Procesando super-resolution... esto puede tardar 20-60 segundos")

# 👇 AJUSTÁ este prompt según tu imagen
# Ejemplos: "a cute dog, high quality", "a person smiling, detailed photo", "a beautiful flower, sharp details"
prompt_descripcion = "high quality photo, detailed"  # 👈 Modificá esto según tu imagen

resultado_upscale = pipe_upscale(
    prompt=prompt_descripcion,
    image=imagen_baja,
    num_inference_steps=30,  # Pasos de difusión (20-50 recomendado)
    guidance_scale=7.5  # Control de adherencia al prompt (modificable)
).images[0]

print("Proceso completado.")

# Comparamos los tres resultados
mostrar_imagenes(
    [imagen_baja, imagen_interpolada, resultado_upscale],
    ['Original Baja (128x128)', 'Interpolación (512x512)', 'Super-Resolution (512x512)']
)

### ¿Qué observamos?

- **Interpolación simple**: La imagen se ve borrosa y pixelada
- **Super-resolution**: El modelo agrega detalles realistas (texturas, bordes definidos)

**Nota**: El resultado no siempre va a ser idéntico al original de alta resolución, pero va a verse mucho mejor que la interpolación simple.

### Experimentación

Probá cambiar los siguientes parámetros en la celda anterior:

**Prompt:**
- Describí específicamente lo que hay en tu imagen
- Ejemplos: "a dog, detailed fur", "a person, sharp facial features", "a flower, detailed petals"
- Un prompt más específico generalmente da mejores resultados

**num_inference_steps:**
- Más pasos = mejor calidad pero más lento
- Probá con 20 (rápido), 30 (moderado), 50 (mejor calidad)

**guidance_scale:**
- Valores más altos siguen más fielmente el prompt
- Probá con 5.0 (más libertad), 7.5 (equilibrado), 12.0 (muy adherente al prompt)

---

## 5. Aplicación 3: Image-to-Image (Modificar características)

### ¿Qué es image-to-image?

Es el proceso de **transformar una imagen existente** manteniendo su estructura básica pero cambiando características específicas. Útil para:
- Cambiar estilos artísticos
- Modificar condiciones (día/noche, verano/invierno)
- Ajustar características manteniendo la composición

### ¿Cómo funciona?

1. Partimos de una imagen existente
2. Agregamos un poco de ruido (controlado por `strength`)
3. El modelo elimina ese ruido siguiendo un prompt
4. El resultado mantiene la estructura pero cambia características

**Parámetro clave - `strength`**:
- `0.0`: sin cambios (imagen original)
- `0.3-0.5`: cambios sutiles (mantiene mucho de la original)
- `0.7-0.9`: cambios significativos (más libertad creativa)
- `1.0`: cambios totales (casi como generar desde cero)

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

# Cargamos el modelo de image-to-image
print("Cargando modelo de image-to-image...")
pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
)
pipe_img2img = pipe_img2img.to(device)
print("Modelo cargado exitosamente.")

### Ejemplo práctico: Transformar un paisaje

In [ ]:
# ============================================
# COMPLETÁ AQUÍ CON TU PROPIA IMAGEN
# ============================================
#
# Tipo de imagen recomendada para IMAGE-TO-IMAGE:
# - Paisaje o escena al aire libre
# - Con cielo visible (para transformaciones climáticas)
# - Ejemplos: campo, playa, montaña, bosque, parque
# - Preferiblemente con buena iluminación
#
# Dónde buscar imágenes:
# - Wikimedia Commons (libres de derechos)
# - Unsplash.com (fotos gratuitas de naturaleza)
# - Pexels.com (fotos gratuitas)
#
# IMPORTANTE: Copiá la URL directa a la imagen (debe terminar en .jpg, .png, etc.)

# Completá tu URL aquí:
url_paisaje = "https://images.unsplash.com/photo-1508881669417-574b54552d73?ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8Nnx8cmFpbnklMjBuYXR1cmV8ZW58MHx8MHx8fDA%3D&fm=jpg&q=60&w=3000"  # 👈 Pegá tu URL entre las comillas

# Cargamos la imagen
imagen_base = cargar_imagen_desde_url(url_paisaje, tamaño=(512, 512))

plt.imshow(imagen_base)
plt.title('Imagen Base')
plt.axis('off')
plt.show()

In [ ]:
# Aplicamos transformación image-to-image
# prompt: descripción de cómo queremos que se vea la imagen
# strength: qué tanto modificar (0.0 = nada, 1.0 = total)
# num_inference_steps: pasos del proceso

print("Procesando transformación... esto puede tardar 10-30 segundos")

# 👇 AJUSTÁ este prompt según la transformación que querés aplicar
# Ejemplos: "sunset with warm colors", "snowy winter scene", "rainy day with clouds", "night scene with stars"
prompt_transformacion = "sunset with warm colors. Retro Polaroid style"  # 👈 Modificá esto según tu transformación deseada

resultado_img2img = pipe_img2img(
    prompt=prompt_transformacion,
    image=imagen_base,
    strength=0.6,  # Cuánto modificar: 0.3=sutil, 0.5=moderado, 0.8=mucho (modificable)
    num_inference_steps=70,  # Pasos del proceso (modificable)
    guidance_scale=7.5  # Adherencia al prompt (modificable)
).images[0]

print("Proceso completado.")

# Comparamos
mostrar_imagenes(
    [imagen_base, resultado_img2img],
    ['Imagen Original', 'Transformada']
)

### Experimentación: Diferentes transformaciones

Probá ejecutar la celda anterior con diferentes prompts y valores de strength.

**Consejos para escribir buenos prompts de transformación:**
- Describí la atmósfera o condición que querés lograr
- Incluí detalles sobre luz, colores, o clima
- Mantené coherencia con el tipo de escena

**Ejemplos de transformaciones climáticas:**
- Atardecer: "sunset with warm orange and pink colors"
- Invierno: "snowy winter landscape with white snow"
- Lluvia: "rainy day with dark clouds and wet ground"
- Noche: "night scene with stars and moonlight"
- Otoño: "autumn scene with orange and red leaves"

**Ejemplos de transformaciones de estilo:**
- Dramático: "dramatic lighting with strong contrast"
- Suave: "soft morning light with pastel colors"
- Vibrante: "vibrant colors and saturated tones"

**Experimentá con strength:**
- `strength=0.3`: cambios sutiles (mantiene casi todo original)
- `strength=0.5`: cambios moderados (equilibrio)
- `strength=0.8`: cambios dramáticos (transformación fuerte)

In [ ]:
# Comparación de diferentes valores de strength
# Ejecutá esto para ver cómo afecta el parámetro strength

print("Generando múltiples versiones con diferentes strength...")
print("Esto va a tardar 1-2 minutos")

valores_strength = [0.3, 0.5, 0.7]

# Usamos el mismo prompt que definiste antes
# Si querés probar otro prompt, modificá la línea siguiente:
prompt_comparacion = prompt_transformacion  # 👈 Usa el prompt anterior, o cambialo por otro

resultados = []
for s in valores_strength:
    resultado = pipe_img2img(
        prompt=prompt_comparacion,
        image=imagen_base,
        strength=s,
        num_inference_steps=30,
        guidance_scale=7.5
    ).images[0]
    resultados.append(resultado)
    print(f"Completado strength={s}")

# Mostramos todas las versiones
mostrar_imagenes(
    [imagen_base] + resultados,
    ['Original', 'Strength=0.3', 'Strength=0.5', 'Strength=0.7']
)

print("\nObservá cómo aumentando strength se producen cambios más dramáticos.")

### ✎ Para pensar

1. Con `strength=0.3` el resultado es muy similar a la imagen original. ¿Qué ocurre matemáticamente para que el modelo preserve tanto la imagen de entrada?
2. ¿Para qué caso de uso profesional usarías image-to-image en lugar de generación puramente desde texto?

---

## 6. Ejercicios prácticos para experimentar

### Ejercicio 1: Inpainting con tu propia imagen

1. Buscá una imagen en internet (o subí la tuya)
2. Creá una máscara simple usando `crear_mascara_rectangular`
3. Probá diferentes prompts para rellenar la región
4. ¿Qué pasa si usás una máscara más grande o más pequeña?

### Ejercicio 2: Comparar super-resolution

1. Tomá una imagen y reducila a diferentes resoluciones (64x64, 96x96, 128x128)
2. Aplicá super-resolution a cada versión
3. Compará los resultados: ¿cuál funciona mejor?

### Ejercicio 3: Explorar transformaciones image-to-image

1. Tomá una imagen de paisaje
2. Probá transformarla a diferentes condiciones climáticas
3. Para cada transformación, probá con tres valores diferentes de strength
4. ¿Cuál valor de strength te parece más útil para cada caso?

### Ejercicio 4: Comparar con métodos tradicionales

Elegí una de las aplicaciones y compará:
- Resultado con modelo de difusión
- Resultado con un método tradicional equivalente (filtros, interpolación, etc.)

¿En qué casos conviene usar modelos de difusión? ¿En qué casos son mejores los métodos tradicionales?

## 7. Parámetros clave: Guía de referencia

### Parámetros comunes en todos los modelos:

| Parámetro | Valores típicos | Efecto | Cuándo modificarlo |
|-----------|-----------------|--------|--------------------|
| `num_inference_steps` | 20-50 | Más pasos = mejor calidad pero más lento | Aumentá para mejor calidad, reducí para velocidad |
| `guidance_scale` | 5-15 | Qué tan fuerte seguir el prompt | Aumentá si el resultado no se parece al prompt, reducí si se ve artificial |
| `prompt` | texto descriptivo | Guía el resultado final | Describí claramente lo que querés ver |

### Parámetros específicos:

**Image-to-Image:**
- `strength` (0.0-1.0): Cuánto modificar la imagen original
  - 0.3: cambios muy sutiles
  - 0.5: equilibrio entre original y transformación
  - 0.8: cambios dramáticos

### Consejos generales:

1. **Empezá con valores moderados**: 30 steps, guidance 7.5, strength 0.5
2. **Cambiá un parámetro a la vez**: para entender su efecto individual
3. **Sé específico en los prompts**: "a sunny beach with blue water" es mejor que "beach"
4. **Si el resultado no es bueno**: probá ejecutar de nuevo (hay aleatoriedad en el proceso)

## 8. Limitaciones y consideraciones

### Ventajas de los modelos de difusión:
- Resultados de muy alta calidad
- Pueden "entender" contenido semántico
- Muy versátiles (múltiples aplicaciones con el mismo modelo)
- No necesitás entrenar nada (usás modelos pre-entrenados)

### Limitaciones:
- **Velocidad**: Mucho más lentos que filtros tradicionales
- **Hardware**: Necesitan GPU para ser razonablemente rápidos
- **Tamaño**: Los modelos ocupan varios GB de espacio
- **Aleatoriedad**: Dos ejecuciones con los mismos parámetros pueden dar resultados ligeramente diferentes
- **Control**: A veces es difícil lograr exactamente lo que querés

### ¿Cuándo usar modelos de difusión vs métodos tradicionales?

**Usá modelos de difusión cuando:**
- Necesitás reconstruir o generar contenido complejo
- La calidad es más importante que la velocidad
- Tenés GPU disponible
- Las operaciones tradicionales no dan buenos resultados

**Usá métodos tradicionales cuando:**
- Necesitás velocidad (tiempo real, procesamiento en lote)
- La operación es simple y predecible
- No tenés GPU o recursos limitados
- Necesitás resultados determinísticos (siempre iguales)

**Lo ideal**: Combinar ambos enfoques según la necesidad.

## 9. Recursos adicionales

### Documentación oficial:
- **Diffusers**: https://huggingface.co/docs/diffusers/
- **Stable Diffusion**: https://stability.ai/

### Para seguir aprendiendo:
- **Hugging Face Diffusion Models Course**: Curso gratuito completo
- **Modelos pre-entrenados**: https://huggingface.co/models?pipeline_tag=text-to-image

### Papers importantes (opcional, para profundizar):
- "Denoising Diffusion Probabilistic Models" (Ho et al., 2020)
- "High-Resolution Image Synthesis with Latent Diffusion Models" (Rombach et al., 2022)

---

## 10. Resumen del módulo

### Lo que aprendimos:

1. **Inpainting**: Eliminar objetos y rellenar regiones de forma coherente
2. **Super-resolution**: Mejorar calidad más allá de la interpolación tradicional
3. **Image-to-image**: Transformar imágenes manteniendo su estructura

### Conceptos clave:

- Los modelos de difusión trabajan en **múltiples pasos iterativos**
- Son **mucho más lentos** que métodos tradicionales
- Producen resultados de **muy alta calidad**
- Necesitan **pocas líneas de código** cuando usamos modelos pre-entrenados
- Los **parámetros modificables** permiten control sobre el resultado

### Próximos pasos:

- Experimentá con tus propias imágenes
- Probá diferentes combinaciones de parámetros
- Explorá otros modelos en Hugging Face
- Pensá en aplicaciones para tus propios proyectos

---

**¿Preguntas o dudas?** Anotá tus consultas para discutirlas en clase.

## ⛰️ Lo que construimos

| Técnica | Para qué se usa | Parámetro clave |
|---|---|---|
| Inpainting | Rellenar regiones con máscara | `mask_image` |
| Image-to-image | Transformar imágenes existentes | `strength` |
| Outpainting | Extender imágenes más allá del borde | `padding` + máscara |

**Próximo cuaderno:** `04_Text_to_Image_SDXL_Turbo` — generamos imágenes desde texto en un solo paso con SDXL-Turbo.